In [13]:
import os

In [5]:
ls

 Volume in drive C has no label.
 Volume Serial Number is 4230-F99E

 Directory of c:\Users\Hp\Code

01/23/2026  09:21 PM    <DIR>          .
01/31/2026  06:51 PM    <DIR>          ..
11/11/2025  11:21 AM    <DIR>          .vscode
01/17/2026  04:52 PM    <DIR>          data-entry-system
01/18/2026  09:44 PM    <DIR>          Forage
01/17/2026  04:32 AM    <DIR>          heart-disease-predicter
02/01/2026  03:16 PM    <DIR>          mlops-chest-cancer-classification
01/04/2026  01:58 AM    <DIR>          NEWSHIT
01/16/2026  03:18 PM    <DIR>          personal
01/16/2026  02:19 AM    <DIR>          ZeroDevAI
               0 File(s)              0 bytes
              10 Dir(s)  240,175,456,256 bytes free


In [ ]:
os.chdir("../")

In [14]:
os.environ["MLFLOW_TRACKING_URI"]="https://dagshub.com/Abdullah2240/mlops-chest-cancer-classification.mlflow" 
os.environ["MLFLOW_TRACKING_USERNAME"]="Abdullah2240" 
os.environ["MLFLOW_TRACKING_PASSWORD"]="73522b937cd582d337b52b8f4120504334323c7e" 

In [15]:
import tensorflow as tf
model = tf.keras.models.load_model("artifacts/training/model.h5")

In [16]:

from dataclasses import dataclass
from pathlib import Path


In [41]:
@dataclass(frozen=True)
class EvaluationConfig():
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int
    params_is_augmentation: bool

In [ ]:
from chestCancerClassifier.utils.common import create_directories, read_yaml, save_json
from chestCancerClassifier import logger
from chestCancerClassifier.constants import *
import mlflow
import mlflow.keras 
from urllib.parse import urlparse

In [44]:
class ConfigurationManager:
    def __init__(self, config_path = CONFIG_FILE_PATH, params_path = PARAMS_FILE_PATH):
        self.config = read_yaml(config_path)
        self.params = read_yaml(params_path)

        create_directories([self.config.artifacts_root])

    def getEvaluationConfig(self):
        eval_config = EvaluationConfig(   
           path_of_model="artifacts/training/model.h5",
           training_data="artifacts/data_ingestion/Chest_CT_scan_Data/",
           all_params=self.params,
           mlflow_uri="https://dagshub.com/Abdullah2240/mlops-chest-cancer-classification.mlflow",
           params_image_size=self.params.IMAGE_SIZE,
           params_batch_size=self.params.BATCH_SIZE,
           params_is_augmentation=self.params.AUGMENTATION)
        return eval_config


In [52]:

class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config
    
    def _valid_generator(self):
        datagenerator_kwargs = dict(
            rescale=1./255,
            validation_split=0.20
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle="False",
            **dataflow_kwargs
        )

        if self.config.params_is_augmentation:
            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range=40,
                horizontal_flip=True,
                width_shift_range=0.2,
                height_shift_range=0.2,
                shear_range=0.2,
                zoom_range=0.2,
                **datagenerator_kwargs
            )
        else:
            train_datagenerator = valid_datagenerator

        self.train_generator = train_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="training",
            shuffle=True,
            **dataflow_kwargs
        )
    
    @staticmethod
    def load_model(path: Path) ->tf.keras.Model:
        return tf.keras.models.load_model(path)

    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        self.score = model.evaluate(self.valid_generator)
        self.save_score()

    def save_score(self):
        scores = {"loss": self.score[0], "accuracy": self.score[1]}
        save_json(path=Path("scores.json"), data=scores)
    
    def log_into_mlflow(self):
        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run():
            mlflow.log_params(self.config.all_params)
            mlflow.log_metrics(
                {"loss": self.score[0], "accuracy": self.score[1]}
            )

            if tracking_url_type_store != "file":
                mlflow.keras.log_model(self.model, "model", registered_model_name="VGG16Model")
            else:
                mlflow.keras.log_model(self.model, "model")
        
        
        

In [ ]:
try:
    config = ConfigurationManager()
    evaluation_config = config.getEvaluationConfig()
    evaluation = Evaluation(config=evaluation_config)
    evaluation.evaluation()
    evaluation.log_into_mlflow()
    
except Exception as e:
    raise e

[2026-02-01 18:55:19,199: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-02-01 18:55:19,211: INFO: common: yaml file: params.yaml loaded successfully]
[2026-02-01 18:55:19,216: INFO: common: created directory at artifacts]
Found 73 images belonging to 2 classes.
Found 296 images belonging to 2 classes.
5/5 [==============================] - 19s 4s/step - loss: 1.4294 - accuracy: 0.8082


2026/02/01 18:55:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/01 18:55:43 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


[2026-02-01 18:55:46,471: WARNING: save: Found untraced functions such as _jit_compiled_convolution_op, _jit_compiled_convolution_op, _jit_compiled_convolution_op, _jit_compiled_convolution_op, _jit_compiled_convolution_op while saving (showing 5 of 14). These functions will not be directly callable after loading.]
[2026-02-01 18:55:47,799: INFO: builder_impl: Assets written to: C:\Users\Hp\AppData\Local\Temp\tmpyk1c1tct\model\data\model\assets]
[2026-02-01 19:00:00,640: WARNING: connectionpool: Retrying (Retry(total=6, connect=7, read=7, redirect=7, status=7)) after connection broken by 'SSLEOFError(8, 'EOF occurred in violation of protocol (_ssl.c:2436)')': /Abdullah2240/mlops-chest-cancer-classification.mlflow/api/2.0/mlflow-artifacts/artifacts/c1d7fb233b09488cb882b9af800f9c3f/models/m-6340ab8c82884aceb7dc5d5df8744190/artifacts/data/model/variables/variables.data-00000-of-00001]


Successfully registered model 'VGG16Model'.
2026/02/01 19:01:51 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: VGG16Model, version 1
Created version '1' of model 'VGG16Model'.


🏃 View run monumental-shark-515 at: https://dagshub.com/Abdullah2240/mlops-chest-cancer-classification.mlflow/#/experiments/0/runs/a34f2a02f6724211a7b2d84d31810ae4
🧪 View experiment at: https://dagshub.com/Abdullah2240/mlops-chest-cancer-classification.mlflow/#/experiments/0
